# 04 — Simulated users

Simulates a user choosing candidates over 120 months (Sections 5.7 and 16) to verify
that Thompson sampling converges on the preferred style arm and that tone counts
steer the `suggestion_probabilities` hint shown in the Create form.

In [ ]:
import sys, pathlib, numpy as np
from collections import Counter
sys.path.insert(0, str(pathlib.Path('..', 'backend').resolve()))

from app.ai.arms import TONES, STRUCTURES, thompson_select, beta_update, suggestion_probabilities

rng = np.random.default_rng(42)
arm_stats: dict[tuple[str, str], tuple[float, float]] = {}
tone_counts: dict[str, int] = {}

favorite = (TONES[1], STRUCTURES[0])      # the arm this simulated user prefers
picks: Counter = Counter()

for month in range(120):
    arms = thompson_select([TONES[0], TONES[1]], arm_stats, rng=rng)
    rates = np.array([0.46 if a == favorite else 0.20 for a in arms])
    chosen = rng.choice(len(arms), p=rates / rates.sum())
    for i, arm in enumerate(arms):
        a, b = arm_stats.get(arm, (1.0, 1.0))
        arm_stats[arm] = beta_update(a, b, "select" if i == chosen else "ignore")
    tone = arms[chosen][0]
    picks[arms[chosen]] += 1
    tone_counts[tone] = tone_counts.get(tone, 0) + 1

print("favorite arm  :", favorite)
print("most picked   :", picks.most_common(3))
print("tone suggestion probs:", {k: round(v, 2) for k, v in suggestion_probabilities(tone_counts).items()})
print("learned beta params  :", {k: arm_stats[k] for k in list(picks)[:3]})